# Analyse musicale FMA – Classification de genres
Inspiré par [tunebat.com](https://tunebat.com)

## Jeu de données – fma_metadata.zip
[https://os.unil.cloud.switch.ch/fma/fma_metadata.zip](https://os.unil.cloud.switch.ch/fma/fma_metadata.zip)

## Membres du groupe Shazam
 - Frederik Kockisch
 - Maya Cugniet
 - Clark Naveau

## Table de matières
à remplir

## Fichiers de FMA utilisés
| Fichier | Contenu |
|---------|--------|
| `features.csv` | 518 caractéristiques audio (chroma, MFCC, ...) |
| `tracks.csv` | Données des musiques : artistes, durée, ... |
| `genres.csv` | Labels genres |

## Partie 1 – Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)

sns.set_theme(style='darkgrid', palette='deep')

RANDOM_STATE = 42
DATA_DIR = 'fma_metadata'

## Partie 2 – Chargement des données

On charge les 3 fichiers `features.csv`, `tracks.csv` et `genres.csv`.

In [ ]:
print('Chargement de features.csv ...')
features = pd.read_csv(f'{DATA_DIR}/features.csv', index_col=0, header=[0, 1, 2])
print(f'  Dimensions : {features.shape}')
print(f'  Groupes de caractéristiques : {features.columns.get_level_values(0).unique().tolist()}')

print('Chargement de tracks.csv ...')
tracks = pd.read_csv(f'{DATA_DIR}/tracks.csv', index_col=0, header=[0, 1, 2])
print(f'  Dimensions : {tracks.shape}')

print('Chargement de genres.csv ...')
genres = pd.read_csv(f'{DATA_DIR}/genres.csv', index_col=0)
print(f'  Dimensions : {genres.shape}')
print('Terminé.')

### 2.1 Description des caractéristiques audio

En réalité, les 518 caractéristiques de `features.csv` peuvent être condensés en 11 groupes. Les données ont été extraites de segments de 30 secondes de morceaux, ainsi les caractéristiques sont détaillées par `mean`, `std`, `skew`, `kurtosis`, `median`, `min` et `max`. Pour notre projet on va seulement s'intéresser aux valeurs `mean` et `std` : la moyenne et l'écart-type des caractéristiques. 

De plus, les 

| Caractéristique | Description | Canaux |
|--------|-------------|--------|
| `chroma_cens` | Énergie par classe de hauteur tonale (C…B) – CENS | 12 notes C=0, C#=1, ... |
| `chroma_cqt` | Chroma via transformée Q constante | 12 C=0, C#=1, ... |
| `chroma_stft` | Chroma via STFT | 12 C=0, C#=1, ... |
| `mfcc` | Mel-Frequency Cepstral Coefficients – timbre spectral | 20 coefficients du moins au plus fin |
| `rmse` | Root-mean-square energy – volume perçu | 1 |
| `spectral_bandwidth` | Largeur de bande spectrale – étalement fréquentiel | 1 |
| `spectral_centroid` | Centroïde spectral – brillance perçue | 1 |
| `spectral_contrast` | Contraste crêtes/vallées – punch tonal | 7 bandes d'un égaliseur audio (sub-bass, bass, low-mid, ...)  |
| `spectral_rolloff` | Fréquence de coupure à 85 % – contenu haute fréquence | 1 |
| `tonnetz` | Relations harmoniques (quinte, tierce…) | 6 relations harmoniques (quinte, tierce mineure, tierce majeure) |
| `zcr` | Taux de passages à zéro – proxy d'activité rythmique | 1 |


### 2.2 Filtrage sur le sous-ensemble medium – suppression des tracks sans label

Avant de commencer à travailler sur le dataset entier, on va filtrer les données. On commence par ne garder que les pistes d'une partie du subset (medium) pour faciliter le travail. On pourra reprendre les données du dataset entier plus tard.

In [ ]:
idx = tracks['set']['subset'].squeeze() == 'medium'
idx = idx[idx].index

feat_large = features.loc[idx]
genre_top = tracks.loc[idx]['track']['genre_top'].squeeze()
duration = tracks.loc[idx]['track']['duration'].squeeze()

mask = genre_top.notna()
feat_large = feat_large[mask]
genre_top = genre_top[mask]
duration = duration[mask]

print(f'{feat_large.shape[0]} tracks, {feat_large.shape[1]} caractéristiques')
print(f'Genres ({genre_top.nunique()}) : {sorted(genre_top.unique())}')

## Partie 3 – Exploration & visualisation des données

### 3.1 Distribution des genres

On vérifie la répartition des genres dans l'ensemble de données. On observe que les genres `Electronic` et `Rock` sont largement majoritaires. Ceci pourrait poser problème lors de la classification, on va devoir se reporter sur un dataset équitable plus tard. Dans un jeu de données *scientifique* on devrait penser à éliminer le bruit des valeurs extrêmes (<1% & >99%), or on travaille sur des oeuvres d'art.

In [ ]:
genre_counts = genre_top.value_counts()

fig, ax = plt.subplots(figsize=(9, 5))

ax.barh(genre_counts.index, genre_counts.values, color=sns.color_palette('deep'))
ax.set_xlabel('Nombre de tracks')
ax.set_title('Distribution des genres')

for i, v in enumerate(genre_counts.values):
    ax.text(v, i, str(v), va='center', fontsize=9)

plt.tight_layout()
plt.savefig('plots/class_distribution.png')
plt.show()

### 3.3 Distribution des groupes de caractéristiques par genre

In [ ]:
plot_feats = {
    'MFCC 1 (timbre)': feat_large['mfcc']['mean']['01'].astype(float),
    'Centroïde spectral': feat_large['spectral_centroid']['mean']['01'].astype(float),
    'ZCR (rythme)': feat_large['zcr']['mean']['01'].astype(float),
}

tmp = pd.DataFrame(plot_feats)
tmp['genre'] = genre_top.values

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, col in enumerate(plot_feats.keys()):
    sns.boxplot(data=tmp, x='genre', y=col, ax=axes[i],
                palette='tab20', showfliers=False)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=45, labelsize=8)
    axes[i].tick_params(axis='y', labelsize=9)
    axes[i].grid(axis='y', alpha=0.4)

plt.suptitle('Features les plus discriminantes par genre')

plt.tight_layout()
plt.savefig('plots/feature_distributions_by_genre.png')
plt.show()

On observe quelques données chevauchantes. On peut s'attendre à des résultats de classification différents de ceux obtenus sur des dataset *toy*. En effet, les genres de musiques n'ont pas nécessairement de définition claire et **ne peuvent pas être précisement différenciés; même par des humains...**

## Partie 4 – Sélection des caractéristiques

| Groupe | Justification |
|--------|---------------|
| `mfcc` | Timbre spectral – le plus discriminant pour le genre musical |
| `chroma_cens` | Contenu tonal – sépare les genres harmoniques (folk, pop) des non-harmoniques |
| `spectral_centroid` | Brillance – élevé pour l'électronique/rock, bas pour l'ambient/folk |
| `spectral_contrast` | Punch tonal – discrimine rock/hip-hop vs ambient |
| `spectral_rolloff` | Fréquence de coupure – corrèle avec le contenu haute fréquence |
| `zcr` | Taux de passage à zéro – proxy d'activité rythmique |
| `rmse` | Amplitude – volume moyen de la track |
| `tonnetz` | Relations harmoniques – contenu tonal avancé |

In [ ]:
le = LabelEncoder()
y = le.fit_transform(genre_top)

selected_groups = {
    'chroma_cens': ['mean', 'std'],
    'chroma_cqt': ['mean', 'std'],
    'chroma_stft': ['mean', 'std'],
    'mfcc': ['mean', 'std'],
    'rmse': ['mean', 'std'],
    'spectral_bandwidth': ['mean', 'std'],
    'spectral_centroid': ['mean', 'std'],
    'spectral_contrast': ['mean', 'std'],
    'spectral_rolloff': ['mean', 'std'],
    'tonnetz': ['mean', 'std'],
    'zcr': ['mean', 'std'],
}

cols = []
for group, stats in selected_groups.items():
    for stat in stats:
        try:
            cols.append(feat_large[group][stat])
        except KeyError:
            print(f'  {group}/{stat} introuvable, ignoré.')

X_all = pd.concat(cols, axis=1).astype(float)
X_arr = np.nan_to_num(X_all.values, nan=0.0, posinf=0.0, neginf=0.0)
print(f'Caractéristiques avant sélection : {X_arr.shape[1]}')

selector = SelectKBest(f_classif, k=100)
X_sel = selector.fit_transform(X_arr, y)
print(f'Caractéristiques après SelectKBest(k=100) : {X_sel.shape[1]}')

feat_names = []
for group, stats in selected_groups.items():
    for stat in stats:
        try:
            n = feat_large[group][stat].shape[1]
            feat_names.extend([group] * n)
        except KeyError:
            pass

from collections import Counter
selected_mask = selector.get_support()
feat_names_arr = [feat_names[i] for i in range(len(feat_names)) if i < len(selected_mask)]
group_counts = Counter([feat_names_arr[i] for i, sel in enumerate(selected_mask) if sel])

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(group_counts.keys(), group_counts.values(), color=sns.color_palette('deep', len(group_counts)))
ax.set_title('Caractéristiques sélectionnées par groupe (SelectKBest k=100)')
ax.set_xlabel('Groupe')
ax.set_ylabel('Nombre de caractéristiques retenues')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## Partie 6 – PCA (Réduction de dimensionnalité)

La **PCA (Principal Component Analysis)** transforme N features corrélées en nouveaux axes indépendants qui capturent le maximum de variance.

**Intuition :** si `chroma_cens.mean` et `chroma_cqt.mean` sont très corrélées, elles portent une information redondante. La PCA les fusionne en une seule composante qui résume les deux.

```
100 features → PCA → ~60 composantes (95% variance préservée)
```

**Seuil 95% :** on garde assez de composantes pour expliquer 95% de la variation dans les données. Les 5% restants = bruit ou information marginale.

**Limite :** les composantes ne sont plus interprétables — « composante 1 » est une combinaison linéaire de toutes les features originales, sans sens musical direct.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sel)

pca = PCA(random_state=RANDOM_STATE)
pca.fit(X_scaled)

cumvar = np.cumsum(pca.explained_variance_ratio_)
n95 = np.searchsorted(cumvar, 0.95) + 1
print(f'Composantes nécessaires pour expliquer 95 % de variance : {n95}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(np.arange(1, len(cumvar) + 1), cumvar, marker='.', ms=3)
axes[0].axhline(0.95, color='red', ls='--', label='seuil 95 %')
axes[0].axvline(n95, color='orange', ls='--', label=f'n={n95}')
axes[0].set_xlabel('Nombre de composantes')
axes[0].set_ylabel('Variance expliquée cumulée')
axes[0].set_title('PCA – Variance cumulée')
axes[0].legend()

axes[1].bar(range(1, 31), pca.explained_variance_ratio_[:30], color=sns.color_palette('deep')[1])
axes[1].set_xlabel('Composante')
axes[1].set_ylabel('Ratio de variance expliquée')
axes[1].set_title('PCA – 30 premières composantes')

plt.tight_layout()
plt.savefig('plots/pca_variance.png', dpi=120, bbox_inches='tight')
plt.show()

pca95 = PCA(n_components=n95, random_state=RANDOM_STATE)
X_pca = pca95.fit_transform(X_scaled)
print(f'Dimensions sortie PCA : {X_pca.shape}')

## Partie 7 – Classificateurs

On entraîne 3 classificateurs et on compare leurs performances sur la classification de genres.

 - **Régression logistique**

 - **Forêt aléatoire** 

 - **SVM** 

### 7.1 Division entraînement / test

In [ ]:
indices = np.arange(len(y))
idx_tr, idx_te = train_test_split(indices, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

X_tr, X_te = X_scaled[idx_tr], X_scaled[idx_te]
Xp_tr, Xp_te = X_pca[idx_tr], X_pca[idx_te]
y_tr, y_te = y[idx_tr], y[idx_te]

print(f'Entraînement : {len(y_tr)} tracks')
print(f'Test         : {len(y_te)} tracks')

### 7.2 Entraînement des classificateurs

In [ ]:
models = {
    'Régression logistique': LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced', random_state=RANDOM_STATE),
    'Forêt aléatoire': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    'SVM (RBF)': SVC(kernel='rbf', C=1.0, class_weight='balanced', random_state=RANDOM_STATE),
}

results = {}
print(f'{"Modèle":<25} {"Acc":>7} {"Préc":>7} {"Rap":>7} {"F1w":>7} {"F1m":>7}')
print('-' * 63)

for name, clf in models.items():
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)

    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_te, y_pred, average='weighted', zero_division=0)
    f1w = f1_score(y_te, y_pred, average='weighted', zero_division=0)
    f1m = f1_score(y_te, y_pred, average='macro', zero_division=0)
    cm = confusion_matrix(y_te, y_pred)

    results[name] = dict(accuracy=acc, precision=prec, recall=rec, f1=f1w, f1_macro=f1m, cm=cm, y_pred=y_pred)
    print(f'{name:<25} {acc:>7.3f} {prec:>7.3f} {rec:>7.3f} {f1w:>7.3f} {f1m:>7.3f}')

### 7.3 Prédictions sur échantillon test

✓ = prédiction correcte, ✗ = erreur.

In [ ]:
np.random.seed(RANDOM_STATE)
sample_idx = sorted(np.random.choice(len(idx_te), size=8, replace=False))
test_ids = feat_large.index[idx_te]
y_pred = results[best_name]['y_pred']
titles = tracks['track']['title'].squeeze()
artists = tracks['artist']['name'].squeeze()

print('=== Prédictions échantillon test ===\n')
for si in sample_idx:
    ti = test_ids[si]
    real = le.inverse_transform([y_te[si]])[0]
    pred = le.inverse_transform([y_pred[si]])[0]
    title = titles.loc[ti] if ti in titles.index else 'N/A'
    artist = artists.loc[ti] if ti in artists.index else 'N/A'
    correct = '✓' if real == pred else '✗'
    print(f'{correct} {ti:>6} | {real:<20} → {pred:<20} | {title} ({artist})')

## Partie 8 – Comparaison des modèles & Conclusion

### 8.1 Comparaison des performances

In [ ]:
results_df = pd.DataFrame({
    name: {k: v for k, v in res.items() if k not in ('cm', 'y_pred')}
    for name, res in results.items()
}).T.astype(float)

print('=== Comparaison finale des modèles ===')
print(results_df.to_string(float_format='{:.4f}'.format))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

metrics = ['accuracy', 'precision', 'recall', 'f1', 'f1_macro']
metrics_labels = ['Exactitude', 'Précision', 'Rappel', 'F1 pondéré', 'F1 macro']
x = np.arange(len(metrics))
width = 0.25
colors = sns.color_palette('deep', len(results_df))

for j, (name, row) in enumerate(results_df.iterrows()):
    axes[0].bar(x + j * width, [row[m] for m in metrics], width, label=name, color=colors[j])

axes[0].set_xticks(x + width)
axes[0].set_xticklabels(metrics_labels, rotation=15, ha='right')
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel('Score')
axes[0].set_title('Comparaison des performances des classificateurs')
axes[0].legend(fontsize=9)
axes[0].axhline(0.5, color='gray', ls='--', alpha=0.4)

results_df['f1_macro'].plot.barh(ax=axes[1], color=colors)
axes[1].set_xlabel('Score F1 macro')
axes[1].set_title('Classement par F1 macro (équilibré)')
axes[1].set_xlim(0, 1)
for i, v in enumerate(results_df['f1_macro']):
    axes[1].text(v + 0.005, i, f'{v:.3f}', va='center')

plt.tight_layout()
plt.savefig('plots/model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## Conclusion

AREMPLIR